# Building an Image Classifier with PyTorch

FashionMNIST workflow generated from a single prompt. Run the cells in order in one kernel.
The scripts share imports and variables; datasets and model checkpoints use paths relative to the working directory.


Generated script: 01_setup.py

In [ ]:
# %%
# Run project scripts in numeric order in the same notebook/kernel namespace.
# Later cells reuse these imports and variables.
import torch
import torchvision
import torchvision.transforms.v2 as T
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import matplotlib.pyplot as plt
import numpy as np
import optuna

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")


Generated script: 02_data.py

In [ ]:
# %%
# Run after 01_setup.py in the same notebook/kernel namespace.
transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

full_train_dataset = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=transform
)
class_names = full_train_dataset.classes

torch.manual_seed(42)
train_dataset, val_dataset = torch.utils.data.random_split(
    full_train_dataset, [55_000, 5_000]
)

batch_size = 32
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False
)

sample_image, sample_label = train_dataset[0]
print(f"Shape: {sample_image.shape}")
print(f"Dtype: {sample_image.dtype}")
print(f"Class: {class_names[sample_label]}")


Generated script: 03_model.py

In [ ]:
# %%
# Run after the previous scripts in the same notebook/kernel namespace.
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),
        )

    def forward(self, x):
        return self.layers(x)


torch.manual_seed(42)
model = ImageClassifier(784, 300, 100, 10).to(device)
xentropy = nn.CrossEntropyLoss()


Generated script: 04_train.py

In [ ]:
# %%
# Run after the previous scripts in the same notebook/kernel namespace.
def evaluate_tm(model, data_loader, metric):
    model_device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    metric.to(model_device)
    metric.reset()
    try:
        with torch.inference_mode():
            for images, labels in data_loader:
                images = images.to(model_device)
                labels = labels.to(model_device)
                metric.update(model(images), labels)
        return metric.compute().item()
    finally:
        metric.reset()
        model.train(was_training)


def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
           n_epochs):
    history = {
        "train_losses": [],
        "train_metrics": [],
        "valid_metrics": [],
    }
    model_device = next(model.parameters()).device
    metric.to(model_device)

    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        total_loss = 0.0
        total_samples = 0

        for images, labels in train_loader:
            images = images.to(model_device)
            labels = labels.to(model_device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            # The criterion returns the mean loss for this batch.
            total_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)
            metric.update(logits.detach(), labels)

        train_loss = total_loss / total_samples
        train_metric = metric.compute().item()
        valid_metric = evaluate_tm(model, valid_loader, metric)

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(valid_metric)
        print(
            f"Epoch {epoch + 1}/{n_epochs} | "
            f"loss: {train_loss:.4f} | "
            f"train accuracy: {train_metric:.4f} | "
            f"valid accuracy: {valid_metric:.4f}"
        )

    return history


optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
n_epochs = 5
history = train2(
    model, optimizer, xentropy, metric, train_loader, valid_loader, n_epochs
)


Generated script: 05_plot_accuracy.py

In [ ]:
# %%
# Run after 04_train.py in the same notebook/kernel namespace.
epochs = range(1, len(history["train_metrics"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_metrics"], marker="o", label="Training accuracy")
plt.plot(epochs, history["valid_metrics"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.xticks(epochs)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Generated script: 06_predict.py

In [ ]:
# %%
# Run after the previous scripts in the same notebook/kernel namespace.
images, labels = next(iter(valid_loader))
images, labels = images[:3], labels[:3]

model.eval()
with torch.no_grad():
    logits = model(images.to(device))
    probabilities = F.softmax(logits, dim=1)
    predictions = probabilities.argmax(dim=1)
    top_probabilities, top_indices = probabilities.topk(4, dim=1)

# Move to CPU before rounding, including when predictions were made on MPS.
probabilities = probabilities.cpu()
predictions = predictions.cpu()
top_probabilities = top_probabilities.cpu()
top_indices = top_indices.cpu()
labels = labels.cpu()
rounded_probabilities = np.round(probabilities.numpy(), decimals=3)

print(f"Probability class order: {class_names}")
for i in range(len(images)):
    print(f"\nImage {i + 1}")
    print(f"Predicted: {class_names[predictions[i].item()]}")
    print(f"True: {class_names[labels[i].item()]}")
    print(f"Softmax probabilities: {rounded_probabilities[i]}")
    print("Top-4 classes:")
    for class_index, probability in zip(top_indices[i], top_probabilities[i]):
        print(f"  {class_names[class_index.item()]}: {probability.item():.3f}")


Generated script: 07_test_eval.py

In [ ]:
# %%
# Run after the previous scripts in the same notebook/kernel namespace.
test_accuracy = evaluate_tm(model, test_loader, metric)
print(f"Test-set accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")


Generated script: 08_save_load.py

In [ ]:
# %%
# Run after the previous scripts in the same notebook/kernel namespace.
from pathlib import Path

model_path = Path("models/my_fashion_mnist_model.pt")
model_path.parent.mkdir(parents=True, exist_ok=True)

hyperparameters = {
    "n_inputs": model.layers[1].in_features,
    "n_hidden1": model.layers[1].out_features,
    "n_hidden2": model.layers[3].out_features,
    "n_classes": model.layers[5].out_features,
}
torch.save(
    {"state_dict": model.state_dict(), "hyperparameters": hyperparameters},
    model_path,
)

checkpoint = torch.load(model_path, map_location=device, weights_only=True)
loaded_model = ImageClassifier(**checkpoint["hyperparameters"]).to(device)
loaded_model.load_state_dict(checkpoint["state_dict"])

model.eval()
loaded_model.eval()
with torch.no_grad():
    prediction_images = images[:3].to(device)
    original_outputs = model(prediction_images)
    loaded_outputs = loaded_model(prediction_images)

predictions_match = torch.allclose(original_outputs, loaded_outputs)
assert predictions_match, "Reloaded model outputs do not match the original model."
print(f"Saved and reloaded model: {model_path}")
print(f"Predictions match on the 3 images: {predictions_match}")


Generated script: 09_optuna.py

In [ ]:
# %%
# Run after the previous scripts in the same notebook/kernel namespace.
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)

    torch.manual_seed(42)
    trial_model = ImageClassifier(784, n_hidden, 100, 10).to(device)
    trial_optimizer = torch.optim.SGD(
        trial_model.parameters(), lr=learning_rate
    )
    trial_criterion = nn.CrossEntropyLoss()
    trial_metric = torchmetrics.Accuracy(
        task="multiclass", num_classes=10
    ).to(device)

    for epoch in range(3):
        print(f"Trial {trial.number}, epoch {epoch + 1}/3")
        trial_history = train2(
            trial_model, trial_optimizer, trial_criterion, trial_metric,
            train_loader, valid_loader, n_epochs=1
        )
        validation_accuracy = trial_history["valid_metrics"][-1]
        trial.report(validation_accuracy, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return validation_accuracy


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    # Allow pruning in a two-trial study after one completed reference trial.
    pruner=optuna.pruners.MedianPruner(n_startup_trials=1),
)
study.optimize(objective, n_trials=2)

print(f"best_value: {study.best_value:.4f}")
print(f"best_params: {study.best_params}")
